# Data Reading

In [0]:
df = spark.read.format('parquet')\
              .option('inferSchema',True)\
                .load('abfss://bronze@azhancarstorage.dfs.core.windows.net/rawdata')

In [0]:
df.display()

# Data Transformation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = df.withColumn('model_category',split(col('Model_ID'),'-')[0])

In [0]:
df = df.withColumn('RevPerUnit',col('Revenue')/col('Units_Sold'))


# Ad-Hoc 

In [0]:
df.groupBy('Year','BranchName').agg(sum('Units_Sold').alias('total_units_sold')).sort('Year','total_units_sold',ascending=[1,0]).display()


Databricks visualization. Run in Databricks to view.

# Handle Nulls

In [0]:

null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.display()


In [0]:
df.count()

In [0]:
df.filter(col('DealerName').isNull()).display()

In [0]:
df.fillna({'DealerName':'Unknown Dealer'}).display()


# Data Writing

In [0]:
df.write.format('parquet')\
        .mode('overwrite')\
        .option('path','abfss://silver@azhancarstorage.dfs.core.windows.net/carsales')\
        .save()

In [0]:
%sql

SELECT * FROM PARQUET.`abfss://silver@azhancarstorage.dfs.core.windows.net/carsales`